In [1]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("../retailiq.db")

# make sure the output folder exists
os.makedirs("../powerbi/data", exist_ok=True)

tables_to_export = {
    "customer_features": "customer_features.csv",
    "cohort_retention": "cohort_retention.csv",
    "rfm_segments": "rfm_segments.csv",
    "delivery_vs_review": "delivery_vs_review.csv",
    "seller_performance": "seller_performance.csv",
    "geo_sales": "geo_sales.csv",
}

for table_name, filename in tables_to_export.items():
    try:
        df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
        df.to_csv(f"../powerbi/data/{filename}", index=False)
        print(f"Exported {table_name} -> {filename} ({len(df)} rows)")
    except Exception as e:
        print(f"FAILED: {table_name} -> {e}")

conn.close()

Exported customer_features -> customer_features.csv (93397 rows)
Exported cohort_retention -> cohort_retention.csv (219 rows)
Exported rfm_segments -> rfm_segments.csv (93357 rows)
Exported delivery_vs_review -> delivery_vs_review.csv (2 rows)
Exported seller_performance -> seller_performance.csv (3036 rows)
Exported geo_sales -> geo_sales.csv (27 rows)


In [2]:
import sqlite3
conn = sqlite3.connect("../retailiq.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())
conn.close()

[('orders',), ('order_items',), ('customers',), ('payments',), ('reviews',), ('products',), ('sellers',), ('geolocation',), ('orders_clean',), ('payments_clean',), ('reviews_clean',), ('customers_clean',), ('order_items_clean',), ('products_clean',), ('sellers_clean',), ('rfm_segments',), ('cohort_retention',), ('repeat_purchase_intervals',), ('delivery_vs_review',), ('customer_features',), ('seller_performance',), ('geo_sales',)]


In [3]:
import sqlite3
conn = sqlite3.connect("../retailiq.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())
conn.close()

[('orders',), ('order_items',), ('customers',), ('payments',), ('reviews',), ('products',), ('sellers',), ('geolocation',), ('orders_clean',), ('payments_clean',), ('reviews_clean',), ('customers_clean',), ('order_items_clean',), ('products_clean',), ('sellers_clean',), ('rfm_segments',), ('cohort_retention',), ('repeat_purchase_intervals',), ('delivery_vs_review',), ('customer_features',), ('seller_performance',), ('geo_sales',)]


In [4]:
import sqlite3
conn = sqlite3.connect("../retailiq.db")
cursor = conn.cursor()

for table in ["order_items_clean", "orders_clean", "customers_clean", "sellers_clean", "reviews_clean", "geolocation"]:
    cursor.execute(f"PRAGMA table_info({table});")
    print(f"\n{table}:")
    for col in cursor.fetchall():
        print(" ", col[1])

conn.close()


order_items_clean:
  order_id
  order_item_id
  product_id
  seller_id
  shipping_limit_date
  price
  freight_value

orders_clean:
  order_id
  customer_id
  order_status
  order_purchase_timestamp
  order_approved_at
  order_delivered_carrier_date
  order_delivered_customer_date
  order_estimated_delivery_date

customers_clean:
  customer_id
  customer_unique_id
  customer_zip_code_prefix
  customer_city
  customer_state

sellers_clean:
  seller_id
  seller_zip_code_prefix
  seller_city
  seller_state

reviews_clean:
  review_id
  order_id
  review_score
  review_comment_title
  review_comment_message
  review_creation_date
  review_answer_timestamp

geolocation:
  geolocation_zip_code_prefix
  geolocation_lat
  geolocation_lng
  geolocation_city
  geolocation_state


In [5]:
import sqlite3
conn = sqlite3.connect("../retailiq.db")
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(customers_clean);")
print("customers_clean:", [col[1] for col in cursor.fetchall()])

cursor.execute("PRAGMA table_info(sellers_clean);")
print("sellers_clean:", [col[1] for col in cursor.fetchall()])

cursor.execute("PRAGMA table_info(reviews_clean);")
print("reviews_clean:", [col[1] for col in cursor.fetchall()])

conn.close()


customers_clean: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
sellers_clean: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
reviews_clean: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [6]:
import sqlite3

conn = sqlite3.connect("../retailiq.db")
with open("../sql/05_seller_performance.sql") as f:
    conn.executescript(f.read())
with open("../sql/06_geo_sales.sql") as f:
    conn.executescript(f.read())
conn.commit()
conn.close()
print("Tables created.")

Tables created.


In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../retailiq.db")
for table_name, filename in {"seller_performance": "seller_performance.csv", "geo_sales": "geo_sales.csv"}.items():
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    df.to_csv(f"../powerbi/data/{filename}", index=False)
    print(f"Exported {table_name} -> {filename} ({len(df)} rows)")
conn.close()

Exported seller_performance -> seller_performance.csv (3036 rows)
Exported geo_sales -> geo_sales.csv (27 rows)


In [8]:
import sqlite3
conn = sqlite3.connect("../retailiq.db")
avg_order = pd.read_sql("SELECT AVG(payment_value) as avg_val FROM payments_clean", conn)
print(avg_order)
conn.close()

      avg_val
0  154.113732


In [9]:
print(df.columns.tolist())

['customer_state', 'total_orders', 'total_revenue', 'avg_order_value']


In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../retailiq.db")
df_check = pd.read_sql("SELECT * FROM customer_features", conn)
conn.close()

print(df_check.columns.tolist())

['customer_unique_id', 'recency_days', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score', 'rfm_total', 'segment', 'repeat_order_count', 'avg_days_between_orders', 'min_days_between_orders', 'max_days_between_orders', 'avg_review_score', 'review_count', 'avg_delivery_days', 'avg_delay_days', 'customer_state', 'churned', 'treatment_voucher']


In [11]:
print(df_check["treatment_voucher"].mean())

0.0388556377613842


In [12]:
conn = sqlite3.connect("retailiq.db")


In [13]:
import os
print(os.getcwd())

e:\RetailIQ\notebooks


In [14]:
import sqlite3
import pandas as pd
import os

# Absolute path, works no matter where the kernel's cwd is
db_path = r"E:\RetailIQ\retailiq.db"
conn = sqlite3.connect(db_path)

avg_order = pd.read_sql("SELECT ROUND(AVG(payment_value), 4) AS avg FROM payments_clean", conn)
total_cust = pd.read_sql("SELECT COUNT(*) AS n FROM customer_features", conn)
voucher_rate = pd.read_sql("SELECT ROUND(AVG(treatment_voucher), 4) AS rate FROM customer_features", conn)

print(avg_order)
print(total_cust)
print(voucher_rate)

conn.close()

        avg
0  154.1137
       n
0  93397
     rate
0  0.0389
